# Post-Cleaning
1. Remove non-music items (based on audio classification with [PANN](https://github.com/qiuqiangkong/audioset_tagging_cnn)
2. De-duplication by file hashes
3. De-duplication by audio fingerprinting


## Load dataset

In [3]:
import os
import json
import torch
import numpy as np
import pandas as pd


def load_dataset(path):
    """
    Load dataset from the specified path.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset not found at {path}")
    
    def inverse_split_dict(split_dict):
        # Add split column to dvi2Torch based on dvi2_torch["split"]
        clique_to_split = {}
        for split_name, sub_dict in split_dict.items():
            for clique in sub_dict.keys():
                clique_to_split[clique] = split_name
        return clique_to_split

    if path.endswith(".json"):
        with open(path, "r") as f:
            meta = json.load(f)
    elif path.endswith(".pt"):
        meta = torch.load(path, weights_only=False)
    if isinstance(meta, dict) and "info" in meta:
        info =  meta["info"]
        split = meta["split"]
    else:
        info, split = meta

    df = pd.DataFrame.from_dict(info, orient="index")
           
    clique2split = inverse_split_dict(split)
    df["split"] = df["clique"].map(clique2split)
    
    if "youtube_id" not in df.columns:
        df["youtube_id"] = df.filename.apply(lambda x: x.split("/")[-1].split(".")[0])
    
    df["dvi"] = ~df.apply(lambda x: x.youtube_id in x.version, axis=1)
    
    return df, meta    

def print_df_summary(df, subset="overall"):
    if subset != "overall":
        df = df.query(f"split == '{subset}'")
    print(f"Summary statistics for {subset}")

    # Column stats
    stats = []
    for col in ["clique", "version", "youtube_id"]:
        unique_count = df[col].nunique()
        total_count = df[col].count()
        stats.append(f"{col:<12} | Unique: {unique_count:>7,} | Total: {total_count:>7,}")
    print("\n".join(stats))

    # DVI distribution
    if "dvi" in df.columns:
        n_true = df["dvi"].sum()
        n_false = (~df["dvi"]).sum()
        total = len(df)
        print("\nDVI distribution:")
        print(f"  True:  {n_true:>7,} ({n_true/total:5.2%})")
        print(f"  False: {n_false:>7,} ({n_false/total:5.2%})")

    # Version per clique stats
    version_per_clique = df.groupby("clique")["version"].nunique()
    print("\nVersion per clique:")
    print(f"  Min: {version_per_clique.min():>3}  | Max: {version_per_clique.max():>3}  | "
          f"Mean: {version_per_clique.mean():>5.2f}  | Median: {version_per_clique.median():>5.2f}  | "
          f"Std: {version_per_clique.std():>5.2f}")
    print("=" * 40)

df, meta = load_dataset("../../data/final/divers.pt")
print_df_summary(df, "train")


Summary statistics for train
clique       | Unique:  78,640 | Total: 1,007,643
version      | Unique: 1,007,643 | Total: 1,007,643
youtube_id   | Unique: 1,007,603 | Total: 1,007,643

DVI distribution:
  True:  328,117 (32.56%)
  False: 679,526 (67.44%)

Version per clique:
  Min:   2  | Max: 470  | Mean: 12.81  | Median:  5.00  | Std: 18.38


## Remove non-musical items

In [4]:
tagging_output_dir = "data/tagging_output_nonmusic_txt"

# List to store filenames without extensions
filenames_no_ext = []

for root, dirs, files in os.walk(tagging_output_dir):
    for file in files:
        # Remove the extension
        name_without_ext = os.path.splitext(file)[0]
        filenames_no_ext.append(name_without_ext)

print(filenames_no_ext)
print(f"Removing {len(filenames_no_ext)} entries from the dataset.")
df = df[~df['youtube_id'].isin(filenames_no_ext)]


[]
Removing 0 entries from the dataset.


## Deduplication
### By Hashes

In [6]:
from collections import defaultdict


def parse_hashed_file(file_path):
    """
    Parses a file containing hashes and associated file paths.
    
    Args:
        file_path (str): Path to the input file.
    """
    # Dictionary to store results
    hash_to_files = defaultdict(list)

    with open(file_path, "r") as f:
        current_hash = None
        for line in f:
            line = line.strip()
            if not line:
                continue  # skip empty lines
            # Accept both "Hash:" and "PCM Hash:" (case-insensitive)
            key, _, value = line.partition(':')
            if key and key.strip().lower() in ("hash", "pcm hash"):
                current_hash = value.strip()
                continue
            elif current_hash is not None:
                # Add file path to the current hash
                hash_to_files[current_hash].append(line)

    # Convert to normal dict if you like
    return dict(hash_to_files)

def get_hashed_inverted(file_path):
    hash_to_files = parse_hashed_file(file_path)
    # Invert hash_to_files: map filename -> hash
    filename_to_hash = {}
    for h, paths in hash_to_files.items():
        for p in paths:
            filename = os.path.splitext(os.path.basename(p))[0]
            filename_to_hash[filename] = h
    return filename_to_hash


# Path to your input file
file_path_basic = "../../data/deduplication/hashed_basic.txt"

hashed = get_hashed_inverted(file_path_basic)

# Now map youtube_id to hash
df['hash'] = df['youtube_id'].map(hashed)
print(f"Number of duplicates based on hash: {df.hash.dropna().duplicated().sum():,}")
print(f"Number of duplicates based on hash: {df.hash.dropna().duplicated().sum() / len(df):.2%}")


Number of duplicates based on hash: 0
Number of duplicates based on hash: 0.00%


#### Conflicting duplicates
Duplicates across different cliques.

In [7]:
df_hash = df.dropna(subset=['hash'])

# get conflicting hashes
grouped = df_hash.groupby('hash')
hashes_conflicting = grouped['clique'].nunique()
hashes_conflicting = hashes_conflicting[hashes_conflicting > 1].index

# filter df
df_conflicting = df_hash[df_hash['hash'].isin(hashes_conflicting)]
df_conflicting = df_conflicting.sort_values(['hash', 'clique'])

# get clique level metadata
df_clique_summary = df.groupby('clique', as_index=False).agg({
    'artist': 'first',  # first non-null in group
    'title': 'first'
})

# merge
cols = ["clique", "version", "youtube_id", "hash"]
df_conflicting = pd.merge(
    df_conflicting[cols].sort_values(['hash', 'clique']),
    df_clique_summary,
    on="clique",
    how="left"
)
df_conflicting

,clique,version,youtube_id,hash,artist,title


#### Deduplication Strategy
We delete by the following strategy: If a hash occurrs across cliques, drop all the versions with that hash. If a hash only occurs in one clique, prioritize an item which is from Discogs (dvi is True). If none is from Discogs, just pick the first to keep and drop the rest of the duplicates by hash. 


In [ ]:
# split df
df_no_hash = df[df['hash'].isna()]
df_with_hash = df[df['hash'].notna()]

# hash counts
hash_clique_counts = df_with_hash.groupby('hash')['clique'].nunique()
hashes_across_cliques = hash_clique_counts[hash_clique_counts > 1].index

# filter
df_filtered = df_with_hash[~df_with_hash['hash'].isin(hashes_across_cliques)].copy()
def pick_row(group):
    # prefer dvi == True
    dvi_true = group[group['dvi'] == True]
    if not dvi_true.empty:
        return dvi_true.iloc[[0]]  # keep first with dvi==True
    else:
        return group.iloc[[0]]  # keep first row
df_deduplicated_hashes = df_filtered.groupby('hash', group_keys=False).apply(pick_row)

# combine
df_dedup = pd.concat([df_deduplicated_hashes, df_no_hash], ignore_index=True)

print(f"After deduplication, dataset size: {len(df_dedup):,}")
print(f"Deduplication removed {len(df) - len(df_dedup):,} rows.")

df_dedup = df_dedup[df_dedup.groupby("clique")["clique"].transform("size") >= 2]
print(f"After singleton removal, dataset size: {len(df_dedup):,}")


/tmp/ipykernel_3274769/1216921341.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_deduplicated_hashes = df_filtered.groupby('hash', group_keys=False).apply(pick_row)


After deduplication, dataset size: 1,349,017
Deduplication removed 0 rows.
After singleton removal, dataset size: 1,349,017


In [ ]:
print_df_summary(df_dedup, "train")
print_df_summary(df_dedup, "valid")
print_df_summary(df_dedup, "test")

Summary statistics for train
clique       | Unique:  78,640 | Total: 1,007,643
version      | Unique: 1,007,643 | Total: 1,007,643
youtube_id   | Unique: 1,007,603 | Total: 1,007,643

DVI distribution:
  True:  328,117 (32.56%)
  False: 679,526 (67.44%)

Version per clique:
  Min:   2  | Max: 470  | Mean: 12.81  | Median:  5.00  | Std: 18.38
Summary statistics for valid
clique       | Unique:   8,736 | Total: 110,587
version      | Unique: 110,587 | Total: 110,587
youtube_id   | Unique: 110,586 | Total: 110,587

DVI distribution:
  True:   35,800 (32.37%)
  False:  74,787 (67.63%)

Version per clique:
  Min:   2  | Max: 266  | Mean: 12.66  | Median:  5.00  | Std: 17.94
Summary statistics for test
clique       | Unique:   9,795 | Total: 230,787
version      | Unique: 230,787 | Total: 230,787
youtube_id   | Unique: 230,785 | Total: 230,787

DVI distribution:
  True:  112,903 (48.92%)
  False: 117,884 (51.08%)

Version per clique:
  Min:   2  | Max: 650  | Mean: 23.56  | Median:  8.00  | 

### Audio Fingerprinting
We use [soundalike](https://codeberg.org/derat/soundalike) which uses the algorithm implemented in  [Chromaprint](https://acoustid.org/chromaprint) to deduplicate a music collection.
```bash
soundalike -db audio.db  data/audio > soundalike_duplicates.txt 2>&1 
```

In [ ]:
def save_df_as_torch(df, out_path, split_col="split"):
    """
    Transform dataframe into 'info' and 'split' dicts, then save as a torch file.

    Args:
        df : pandas.DataFrame
            Must have columns: ['clique', 'version', split_col, ...]
        out_path : str
            Path to save .pt file
        split_col : str
            Column name to drop from 'info' dict (default: "split")
    Returns:
        dict with keys: 'info', 'split'
    """
    # 1. Build unique key = clique:version
    index = df["clique"].astype(str) + ":" + df["version"].astype(str)

    # 2. Build info_dict: key = clique:version, value = row dict excluding split_col
    drop_cols = [split_col] if split_col in df.columns else []
    info_df = df.drop(columns=drop_cols)
    info_dict = {
        idx: dict(zip(info_df.columns, row))
        for idx, row in zip(index, info_df.itertuples(index=False, name=None))
    }

    # 3. Build split_dict: split -> {clique: [versions]}
    split_dict = {}
    if split_col in df.columns:
        # Only include rows where split_col is not NaN
        for split, sub_df in df.dropna(subset=[split_col]).groupby(split_col):
            # Each split maps to a dict of cliques -> list of versions
            clique_dict = sub_df.groupby("clique")["version"].apply(list).to_dict()
            split_dict[split] = clique_dict

    # 4. Save torch file
    data = {"info": info_dict, "split": split_dict}
    torch.save(data, out_path)
    print(f"Saved torch file with keys: {list(data.keys())} to {out_path}")
    return data

In [39]:
import re

soundalike_log_path = "../../data/deduplication/soundalike_duplicates.txt"

def parse_soundalike_log(filename):
    groups = []        # final list of lists
    current_group = [] # current group of tuples
    pattern = re.compile(r"^(.*?)\s+([\d.]+) MB\s+([\d.]+) sec$")

    # Step 1: keep only lines after last "Finished scanning ... files"
    with open(filename, "r", encoding="utf-8") as f:
        lines = f.readlines()

    last_finished_idx = -1
    for i, line in enumerate(lines):
        if "Finished scanning" in line:
            last_finished_idx = i

    lines = lines[last_finished_idx + 1 :]  # only lines after last Finished scanning

    # Step 2: parse lines into groups
    for line in lines:
        line = line.strip()
        if not line:  # blank line → start new group
            if current_group:
                groups.append(current_group)
                current_group = []
            continue

        m = pattern.match(line)
        if m:
            path, size, time_sec = m.groups()
            current_group.append((path.split("/")[1].split(".")[0], float(size), float(time_sec)))

    # Add last group if any
    if current_group:
        groups.append(current_group)
        
    # make df
    rows = []
    for i, group in enumerate(groups):
        for youtube_id, size, duration in group:
            rows.append({
                "group": i,
                "youtube_id": youtube_id,
                "size_mb": size,
                "duration": duration
            })
    df_rows = pd.DataFrame(rows)
    return df_rows

# Usage:
df_soundalike = parse_soundalike_log(soundalike_log_path)

total_items = len(df_soundalike)
num_groups = df_soundalike.group.nunique()
num_duplicates = total_items - num_groups

print(f"Total items: {total_items:,}")
print(f"Number of groups: {num_groups:,}")
print(f"Number of duplicates: {num_duplicates:,}")

if len(df_soundalike) == df_soundalike.youtube_id.nunique():
    print("Consistency Check passed: No overlapping youtube_ids between groups.")
else:
    print("Consistency Check failed: Overlapping youtube_ids found between groups.")
    

Total items: 152,101
Number of groups: 51,433
Number of duplicates: 100,668
Consistency Check passed: No overlapping youtube_ids between groups.


In [ ]:
# merge duplicate group to main df
df = df.merge(df_soundalike[["youtube_id", "group"]], on="youtube_id", how="left")

# Step 1: find groups with more than one unique clique
conflict_groups = df.groupby("group")["clique"].nunique()
conflict_groups = conflict_groups[conflict_groups > 1].index  # groups with conflicts

# Step 2: select all rows in those groups
conflicting_rows = df[df["group"].isin(conflict_groups)]

print(f"Number of conflicting rows: {len(conflicting_rows):,}")


Number of conflicting rows: 10,807


#### Deduplication Strategy
It is the same, like for the hash-based deduplication (see above).


In [ ]:
import pandas as pd

def deduplicate(df):
    """
    Deduplicate a DataFrame according to the following strategy:

    1. If a group has conflicting cliques (more than 1 unique clique), drop all rows in that group.
    2. If a group has only one clique:
       - Keep the row with dvi == True if any exist
       - Otherwise, keep the first row
    3. Rows with group == NaN are kept untouched.
    """
    # Step 0: separate rows without a group
    df_no_group = df[df["group"].isna()].copy()

    # Step 1: consider only rows with valid group
    df_valid = df.dropna(subset=["group"]).copy()

    # Step 2: find groups with more than 1 unique clique
    conflict_groups = (
        df_valid.groupby("group")["clique"].nunique()
        .loc[lambda x: x > 1]
        .index
    )

    # Step 3: drop all rows in conflicting groups
    df_clean = df_valid[~df_valid["group"].isin(conflict_groups)].copy()

    # Step 4: within each non-conflicting group, pick one row
    def pick_one(group_df):
        # Prefer a row with dvi == True
        dvi_rows = group_df[group_df["dvi"] == True]
        if not dvi_rows.empty:
            return dvi_rows.iloc[[0]]  # keep first DVI
        else:
            return group_df.iloc[[0]]  # keep first row if no DVI

    df_dedup = df_clean.groupby("group", group_keys=False).apply(pick_one)

    # Step 5: combine with rows that had no group
    df_final = pd.concat([df_dedup, df_no_group], ignore_index=True)

    return df_final

# Usage:
df_dedup = deduplicate(df)
print(f"Original rows: {len(df):,}, After deduplication: {len(df_dedup):,}")

df_dedup = df_dedup[df_dedup.groupby("clique")["clique"].transform("size") >= 2]
print(f"After singleton removal, dataset size: {len(df_dedup):,}")


/tmp/ipykernel_3274769/1631370867.py:38: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_dedup = df_clean.groupby("group", group_keys=False).apply(pick_one)


Original rows: 1349017, After deduplication: 1246719


In [59]:
print_df_summary(df_dedup, "train")
print_df_summary(df_dedup, "valid")
print_df_summary(df_dedup, "test")


Summary statistics for train
clique       | Unique:  78,138 | Total: 926,582
version      | Unique: 926,582 | Total: 926,582
youtube_id   | Unique: 926,546 | Total: 926,582

DVI distribution:
  True:  324,823 (35.06%)
  False: 601,759 (64.94%)

Version per clique:
  Min:   2  | Max: 465  | Mean: 11.86  | Median:  5.00  | Std: 17.04
Summary statistics for valid
clique       | Unique:   8,695 | Total: 101,571
version      | Unique: 101,571 | Total: 101,571
youtube_id   | Unique: 101,570 | Total: 101,571

DVI distribution:
  True:   35,466 (34.92%)
  False:  66,105 (65.08%)

Version per clique:
  Min:   2  | Max: 264  | Mean: 11.68  | Median:  5.00  | Std: 16.64
Summary statistics for test
clique       | Unique:   9,762 | Total: 217,998
version      | Unique: 217,998 | Total: 217,998
youtube_id   | Unique: 217,996 | Total: 217,998

DVI distribution:
  True:  111,814 (51.29%)
  False: 106,184 (48.71%)

Version per clique:
  Min:   2  | Max: 643  | Mean: 22.33  | Median:  8.00  | Std: 35.41

## Save

In [68]:
def save_df_as_torch(df, out_path, split_col="split"):
    """
    Transform dataframe into 'info' and 'split' dicts, then save as a torch file.

    Args:
        df : pandas.DataFrame
            Must have columns: ['clique', 'version', split_col, ...]
        out_path : str
            Path to save .pt file
        split_col : str
            Column name to drop from 'info' dict (default: "split")
    Returns:
        dict with keys: 'info', 'split'
    """
    # 1. Build unique key = clique:version
    index = df["clique"].astype(str) + ":" + df["version"].astype(str)

    # 2. Build info_dict: key = clique:version, value = row dict excluding split_col
    drop_cols = [split_col] if split_col in df.columns else []
    info_df = df.drop(columns=drop_cols)
    info_dict = {
        idx: dict(zip(info_df.columns, row))
        for idx, row in zip(index, info_df.itertuples(index=False, name=None))
    }

    # 3. Build split_dict: split -> {clique: [versions]}
    split_dict = {}
    if split_col in df.columns:
        # Only include rows where split_col is not NaN
        for split, sub_df in df.dropna(subset=[split_col]).groupby(split_col):
            # Each split maps to a dict of cliques -> list of versions
            clique_dict = sub_df.groupby("clique")["version"].apply(list).to_dict()
            split_dict[split] = clique_dict

    # 4. Save torch file
    data = {"info": info_dict, "split": split_dict}
    torch.save(data, out_path)
    print(f"Saved torch file with keys: {list(data.keys())} to {out_path}")
    return data

save_df_as_torch(df_dedup.drop(columns=["group"]), "../../data/final/divers.pt")


Saved torch file with keys: ['info', 'split'] to ../../data/final/divers.pt


{'info': {'C-0141069:V-1236232': {'id': 915453,
   'clique': 'C-0141069',
   'version': 'V-1236232',
   'artist': 'Jacques Brel',
   'title': 'Les Pieds Dans Le Ruisseau',
   'filename': 'OB/OBqy3jopPLs.mp3',
   'samplerate': 44100,
   'channels': 2,
   'youtube_id': 'OBqy3jopPLs',
   'dvi': True,
   'track_writer_names': ['Jacques Brel'],
   'release_artist_names': ['Jacques Brel'],
   'release_genres': ['Pop'],
   'release_styles': [['Pop', 'Chanson']],
   'country': 'France',
   'labels': ['Philips'],
   'formats': ['Vinyl'],
   'released': '1957',
   'yt_title': 'Les Pieds Dans le Ruisseau',
   'yt_description': 'Provided to YouTube by The Orchard Enterprises\n\nLes Pieds Dans le Ruisseau · Jacques Brel\n\nAll Time Classics\n\n℗ 2011 Sleeping Giant Music\n\nReleased on: 2011-11-07\n\nAuto-generated by YouTube.',
   'yt_tags': array(['Jacques', 'Brel', 'All', 'Time', 'Classics', 'Les', 'Pieds',
          'Dans', 'le', 'Ruisseau'], dtype=object),
   'yt_categories': array(['Music'], 